# 02 - Data Cleaning
Fix each issue found in EDA, with a before/after check. Output: `data/clean.csv`
(train + test stacked, with a `src` column).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "train_v9rqX0R.csv").exists():
    ROOT = ROOT.parent
OUT = ROOT / "data"
OUT.mkdir(exist_ok=True)

train = pd.read_csv(ROOT / "data" / "train_v9rqX0R.csv").assign(src="train")
test = pd.read_csv(ROOT / "data" / "test_AbJTz2l.csv").assign(src="test")
df = pd.concat([train, test], ignore_index=True)   # clean both sets identically
print("combined:", df.shape)
print("missing before:\n", df.drop(columns="Item_Outlet_Sales").isna().sum()[lambda s: s > 0])

combined: (14204, 13)
missing before:
 Item_Weight    2439
Outlet_Size    4016
dtype: int64


## 1. Standardise `Item_Fat_Content`

In [2]:
print("before:", df.Item_Fat_Content.unique())
df["Item_Fat_Content"] = df["Item_Fat_Content"].replace(
    {"LF": "Low Fat", "low fat": "Low Fat", "reg": "Regular"})
print("after: ", df.Item_Fat_Content.unique())

before: <StringArray>
['Low Fat', 'Regular', 'low fat', 'LF', 'reg']
Length: 5, dtype: str
after:  <StringArray>
['Low Fat', 'Regular']
Length: 2, dtype: str


## 2. `Item_Weight` - same item always has the same weight
Fill from the item's other rows; fall back to the item-type mean.

In [3]:
df["Item_Weight"] = df["Item_Weight"].fillna(
    df.groupby("Item_Identifier")["Item_Weight"].transform("mean"))
df["Item_Weight"] = df["Item_Weight"].fillna(
    df.groupby("Item_Type")["Item_Weight"].transform("mean"))
print("Item_Weight missing:", df.Item_Weight.isna().sum())

Item_Weight missing: 0


## 3. `Outlet_Size` - impute with the mode of the outlet type

In [4]:
print(pd.crosstab(df.Outlet_Type, df.Outlet_Size.fillna("MISSING")))
df["Outlet_Size"] = df["Outlet_Size"].fillna(
    df.groupby("Outlet_Type")["Outlet_Size"].transform(lambda s: s.mode()[0]))
print("Outlet_Size missing:", df.Outlet_Size.isna().sum())

Outlet_Size        High  MISSING  Medium  Small
Outlet_Type                                    
Grocery Store         0      925       0    880
Supermarket Type1  1553     3091    1550   3100
Supermarket Type2     0        0    1546      0
Supermarket Type3     0        0    1559      0
Outlet_Size missing: 0


## 4. `Item_Visibility` - 0 is impossible (item is on shelf); treat as missing

In [5]:
print("zeros before:", (df.Item_Visibility == 0).sum())
df["Item_Visibility"] = df["Item_Visibility"].replace(0, np.nan)
df["Item_Visibility"] = df["Item_Visibility"].fillna(
    df.groupby("Item_Identifier")["Item_Visibility"].transform("mean"))
print("zeros/missing after:", df.Item_Visibility.isna().sum())

zeros before: 879
zeros/missing after: 0


## Validate & save

In [6]:
assert df.drop(columns="Item_Outlet_Sales").isna().sum().sum() == 0
df.to_csv(OUT / "clean.csv", index=False)
print("saved", OUT / "clean.csv", df.shape)
df.head()

saved C:\Users\sajss\AppData\Local\Temp\bm_clone\data\clean.csv (14204, 13)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,src
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,train
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,train
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,train
3,FDX07,19.20,Regular,0.022930,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,train
4,NCD19,8.93,Low Fat,0.014670,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,train
